In [1]:
import pandas as pd
import numpy as np

# Load your clean CSVs
matches  = pd.read_csv("../data/clean/matches_clean.csv")
rankings = pd.read_csv("../data/clean/ranking_clean.csv")
wc2026   = pd.read_csv("../data/clean/wc2026_structure.csv")

# Parse dates
matches["date"]       = pd.to_datetime(matches["date"])
rankings["rank_date"] = pd.to_datetime(rankings["rank_date"])

# Sanity checks
print("=== MATCHES ===")
print(matches.shape)
print(matches.columns.tolist())
print(matches["outcome"].unique())
print(matches.head(3))

print("\n=== RANKINGS ===")
print(rankings.shape)
print(rankings.columns.tolist())
print(rankings.head(3))

print("\n=== WC 2026 ===")
print(wc2026.shape)
print(wc2026.columns.tolist())
print(wc2026.head(3))

=== MATCHES ===
(23995, 10)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'outcome']
['home_win' 'draw' 'away_win']
        date            home_team away_team  home_score  away_score  \
0 2000-01-04                Egypt      Togo           2           1   
1 2000-01-07              Tunisia      Togo           7           0   
2 2000-01-08  Trinidad And Tobago    Canada           0           0   

  tournament           city              country  neutral   outcome  
0   Friendly          Aswan                Egypt    False  home_win  
1   Friendly          Tunis              Tunisia    False  home_win  
2   Friendly  Port of Spain  Trinidad and Tobago    False      draw  

=== RANKINGS ===
(54683, 5)
['rank_date', 'fifa_rank', 'team', 'fifa_points', 'confederation']
   rank_date  fifa_rank         team  fifa_points confederation
0 2003-01-15        204  Afghanistan          7.0           AFC
1 2003-02-19        203  Afghanist

In [2]:
# =============================================================
# FEATURE 1 & 2 — FIFA points + ranking point difference
# =============================================================
# For each match, attach the most recent FIFA points for both
# teams published ON OR BEFORE the match date (no leakage).
# =============================================================

# Sort both by date — required by merge_asof
matches  = matches.sort_values("date").reset_index(drop=True)
rankings = rankings.sort_values("rank_date").reset_index(drop=True)

# Keep only what we need from rankings
rankings_slim = rankings[["rank_date", "team", "fifa_points"]].copy()

# --- Merge for HOME team ---
home_merge = pd.merge_asof(
    left      = matches[["date", "home_team"]].rename(columns={"home_team": "team"}),
    right     = rankings_slim,
    left_on   = "date",
    right_on  = "rank_date",
    by        = "team",
    direction = "backward"   # most recent ranking <= match date
).rename(columns={"fifa_points": "home_fifa_points"})

# --- Merge for AWAY team ---
away_merge = pd.merge_asof(
    left      = matches[["date", "away_team"]].rename(columns={"away_team": "team"}),
    right     = rankings_slim,
    left_on   = "date",
    right_on  = "rank_date",
    by        = "team",
    direction = "backward"
).rename(columns={"fifa_points": "away_fifa_points"})

# --- Attach back to matches ---
matches["home_fifa_points"] = home_merge["home_fifa_points"].values
matches["away_fifa_points"] = away_merge["away_fifa_points"].values

# --- Feature 2: difference ---
matches["fifa_points_diff"] = matches["home_fifa_points"] - matches["away_fifa_points"]

# --- Check ---
print(matches[["date", "home_team", "away_team",
               "home_fifa_points", "away_fifa_points",
               "fifa_points_diff"]].head(10))

print("\nNaN home_fifa_points:", matches["home_fifa_points"].isna().sum())
print("NaN away_fifa_points:", matches["away_fifa_points"].isna().sum())

        date            home_team away_team  home_fifa_points  \
0 2000-01-04                Egypt      Togo               NaN   
1 2000-01-07              Tunisia      Togo               NaN   
2 2000-01-08  Trinidad And Tobago    Canada               NaN   
3 2000-01-09         Burkina Faso     Gabon               NaN   
4 2000-01-09            Guatemala   Armenia               NaN   
5 2000-01-09          Ivory Coast     Egypt               NaN   
6 2000-01-09               Mexico      Iran               NaN   
7 2000-01-11              Bermuda    Canada               NaN   
8 2000-01-11         Burkina Faso  Cameroon               NaN   
9 2000-01-13              Senegal  Cameroon               NaN   

   away_fifa_points  fifa_points_diff  
0               NaN               NaN  
1               NaN               NaN  
2               NaN               NaN  
3               NaN               NaN  
4               NaN               NaN  
5               NaN               NaN  
6   

In [3]:
# =============================================================
# DIAGNOSE — how many matches fall before ranking data starts?
# =============================================================

print("Earliest ranking date:", rankings["rank_date"].min())
print("Earliest match date:  ", matches["date"].min())

# How many matches are before the first ranking date?
first_rank_date = rankings["rank_date"].min()
pre_ranking_matches = matches[matches["date"] < first_rank_date]
print(f"\nMatches before first ranking: {len(pre_ranking_matches)}")
print(f"Date range: {pre_ranking_matches['date'].min()} → {pre_ranking_matches['date'].max()}")

# Check if there are NaNs AFTER the ranking data starts (would be a real problem)
post_ranking_matches = matches[matches["date"] >= first_rank_date]
nan_after = post_ranking_matches[["home_fifa_points", "away_fifa_points"]].isna().sum()
print(f"\nNaNs AFTER first ranking date (should be small):")
print(nan_after)

# Which teams cause NaNs after the ranking starts?
problem_rows = post_ranking_matches[
    post_ranking_matches["home_fifa_points"].isna() |
    post_ranking_matches["away_fifa_points"].isna()
]
missing_home = problem_rows[problem_rows["home_fifa_points"].isna()]["home_team"].unique()
missing_away = problem_rows[problem_rows["away_fifa_points"].isna()]["away_team"].unique()
print(f"\nTeams missing from rankings (home): {missing_home}")
print(f"Teams missing from rankings (away): {missing_away}")

# =============================================================
# FIX — fill NaNs with the global minimum fifa_points
# Reasoning: unknown teams are weakest, min is a safe lower bound
# =============================================================

min_points = rankings["fifa_points"].min()
print(f"\nFilling NaNs with global min fifa_points: {min_points}")

matches["home_fifa_points"] = matches["home_fifa_points"].fillna(min_points)
matches["away_fifa_points"] = matches["away_fifa_points"].fillna(min_points)
matches["fifa_points_diff"] = matches["home_fifa_points"] - matches["away_fifa_points"]

# Confirm zero NaNs
print("\nNaN home_fifa_points after fix:", matches["home_fifa_points"].isna().sum())
print("NaN away_fifa_points after fix:", matches["away_fifa_points"].isna().sum())

# Final check
print("\n--- Sample after fix ---")
print(matches[["date", "home_team", "away_team",
               "home_fifa_points", "away_fifa_points",
               "fifa_points_diff"]].head(5))

Earliest ranking date: 2000-01-19 00:00:00
Earliest match date:   2000-01-04 00:00:00

Matches before first ranking: 15
Date range: 2000-01-04 00:00:00 → 2000-01-18 00:00:00

NaNs AFTER first ranking date (should be small):
home_fifa_points    1943
away_fifa_points    2186
dtype: int64

Teams missing from rankings (home): ['Dr Congo' 'United States Virgin Islands' 'Saint Kitts And Nevis'
 'Bhutan' 'Kernow' 'Saint Vincent And The Grenadines' 'Saint Lucia'
 'Curaçao' 'Gambia' 'Alderney' 'Serbia' 'New Caledonia'
 'São Tomé And Príncipe' 'Cape Verde' 'Jersey' 'Kyrgyzstan' 'Gibraltar'
 'Zanzibar' 'Catalonia' 'Andalusia' 'Basque Country' 'Saint Martin'
 'Martinique' 'Brunei' 'Guadeloupe' 'Taiwan' 'Guernsey' 'Greenland'
 'Ynys Môn' 'Isle Of Wight' 'Isle Of Man' 'Saare County' 'Orkney' 'Rhodes'
 'Monaco' 'North Korea' 'Canary Islands' 'French Guiana' 'Afghanistan'
 'Shetland' 'Frøya' 'Sark' 'Gotland' 'Tuvalu' 'Micronesia' 'Kiribati'
 'Mayotte' 'Réunion' 'Comoros' 'Åland Islands' 'Timor-Leste' 

In [4]:
# =============================================================
# CONFIRM — zero NaNs after fill
# =============================================================

assert matches["home_fifa_points"].isna().sum() == 0
assert matches["away_fifa_points"].isna().sum() == 0
assert matches["fifa_points_diff"].isna().sum() == 0

print("✅ Features 1 & 2 done. No NaNs remaining.")
print(f"   home_fifa_points range: {matches['home_fifa_points'].min():.1f} → {matches['home_fifa_points'].max():.1f}")
print(f"   away_fifa_points range: {matches['away_fifa_points'].min():.1f} → {matches['away_fifa_points'].max():.1f}")
print(f"   fifa_points_diff range: {matches['fifa_points_diff'].min():.1f} → {matches['fifa_points_diff'].max():.1f}")

✅ Features 1 & 2 done. No NaNs remaining.
   home_fifa_points range: 0.0 → 2164.0
   away_fifa_points range: 0.0 → 2164.0
   fifa_points_diff range: -1775.0 → 1838.4


In [5]:
# =============================================================
# FEATURE 3 — Recent form (win rate in last 10 matches)
# FEATURE 6 — Avg goal difference in last 10 matches
# =============================================================
# Key rule: shift(1) before rolling so the current match is
# NEVER included in its own feature calculation (no leakage).
# =============================================================

# We need to see each match from BOTH teams' perspectives.
# Build a "long" table: one row per team per match.

home_view = pd.DataFrame({
    "match_idx" : matches.index,
    "date"      : matches["date"].values,
    "team"      : matches["home_team"].values,
    "win"       : (matches["outcome"] == "home_win").astype(int).values,
    "goal_diff" : (matches["home_score"] - matches["away_score"]).values,
})

away_view = pd.DataFrame({
    "match_idx" : matches.index,
    "date"      : matches["date"].values,
    "team"      : matches["away_team"].values,
    "win"       : (matches["outcome"] == "away_win").astype(int).values,
    "goal_diff" : (matches["away_score"] - matches["home_score"]).values,
})

# Stack into one long table, sort by team + date
team_view = pd.concat([home_view, away_view], ignore_index=True)
team_view = team_view.sort_values(["team", "date", "match_idx"]).reset_index(drop=True)

# shift(1) + rolling(10) per team — no leakage
team_view["form"] = (
    team_view.groupby("team")["win"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

team_view["avg_goal_diff"] = (
    team_view.groupby("team")["goal_diff"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

# First match of every team will be NaN (nothing before it to shift)
# Fill with neutral values: 0.5 win rate, 0.0 goal diff
team_view["form"]          = team_view["form"].fillna(0.5)
team_view["avg_goal_diff"] = team_view["avg_goal_diff"].fillna(0.0)

# Split back into home and away using match_idx
home_stats = team_view[team_view["match_idx"].isin(matches.index) &
                        (team_view["team"] == team_view["match_idx"].map(matches["home_team"]))
                       ].set_index("match_idx").sort_index()

away_stats = team_view[team_view["match_idx"].isin(matches.index) &
                        (team_view["team"] == team_view["match_idx"].map(matches["away_team"]))
                       ].set_index("match_idx").sort_index()

matches["home_form"]          = home_stats["form"].values
matches["away_form"]          = away_stats["form"].values
matches["home_avg_goal_diff"] = home_stats["avg_goal_diff"].values
matches["away_avg_goal_diff"] = away_stats["avg_goal_diff"].values

# Check
print(matches[["date", "home_team", "away_team",
               "home_form", "away_form",
               "home_avg_goal_diff", "away_avg_goal_diff"]].head(10))

print("\nNaNs:")
print(matches[["home_form","away_form",
               "home_avg_goal_diff","away_avg_goal_diff"]].isna().sum())

        date            home_team away_team  home_form  away_form  \
0 2000-01-04                Egypt      Togo        0.5        0.5   
1 2000-01-07              Tunisia      Togo        0.5        0.0   
2 2000-01-08  Trinidad And Tobago    Canada        0.5        0.5   
3 2000-01-09         Burkina Faso     Gabon        0.5        0.5   
4 2000-01-09            Guatemala   Armenia        0.5        0.5   
5 2000-01-09          Ivory Coast     Egypt        0.5        1.0   
6 2000-01-09               Mexico      Iran        0.5        0.5   
7 2000-01-11              Bermuda    Canada        0.5        0.0   
8 2000-01-11         Burkina Faso  Cameroon        0.0        0.5   
9 2000-01-13              Senegal  Cameroon        0.5        0.0   

   home_avg_goal_diff  away_avg_goal_diff  
0                 0.0                 0.0  
1                 0.0                -1.0  
2                 0.0                 0.0  
3                 0.0                 0.0  
4                 0.

In [6]:
# =============================================================
# FEATURE 4 — Head-to-head win rate
# =============================================================
# For each match, what fraction of past meetings did the
# home team win against this specific away team?
#
# Win  → 1.0  |  Draw → 0.5  |  Loss → 0.0
# No prior meetings → 0.5 (neutral, no information)
#
# shift(1) before expanding mean = no leakage.
# =============================================================

# Encode result from home team's perspective
def encode_outcome(outcome):
    if outcome == "home_win": return 1.0
    if outcome == "draw":     return 0.5
    return 0.0

matches["_home_result"] = matches["outcome"].map(encode_outcome)

# expanding().mean() after shift(1) gives the historical average
# of all matches between this exact home/away pair before today
matches["h2h_home_win_rate"] = (
    matches.groupby(["home_team", "away_team"])["_home_result"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# First meeting between any pair → NaN → fill with 0.5
matches["h2h_home_win_rate"] = matches["h2h_home_win_rate"].fillna(0.5)

# Drop the temp column
matches.drop(columns=["_home_result"], inplace=True)

# Check
print(matches[["date", "home_team", "away_team",
               "h2h_home_win_rate"]].head(10))

print("\nNaNs:", matches["h2h_home_win_rate"].isna().sum())
print(f"Range: {matches['h2h_home_win_rate'].min():.2f} → {matches['h2h_home_win_rate'].max():.2f}")

        date            home_team away_team  h2h_home_win_rate
0 2000-01-04                Egypt      Togo                0.5
1 2000-01-07              Tunisia      Togo                0.5
2 2000-01-08  Trinidad And Tobago    Canada                0.5
3 2000-01-09         Burkina Faso     Gabon                0.5
4 2000-01-09            Guatemala   Armenia                0.5
5 2000-01-09          Ivory Coast     Egypt                0.5
6 2000-01-09               Mexico      Iran                0.5
7 2000-01-11              Bermuda    Canada                0.5
8 2000-01-11         Burkina Faso  Cameroon                0.5
9 2000-01-13              Senegal  Cameroon                0.5

NaNs: 0
Range: 0.00 → 1.00


In [7]:
# =============================================================
# FEATURE 5 — Tournament weight
# =============================================================
# Reflects how important the match is.
# Higher weight = more meaningful result for predicting WC.
# =============================================================

print("All tournament types in dataset:")
print(matches["tournament"].value_counts())

All tournament types in dataset:
tournament
Friendly                                8020
FIFA World Cup qualification            5332
UEFA Euro qualification                 1532
African Cup of Nations qualification    1416
UEFA Nations League                      630
                                        ... 
ConIFA Challenger Cup                      1
The Other Final                            1
Copa Confraternidad                        1
East Asian Games                           1
Benedikt Fontana Cup                       1
Name: count, Length: 112, dtype: int64


In [8]:
# =============================================================
# FEATURE 5 — Tournament weight
# =============================================================
# Scale: 1.0 (friendlies) → 5.0 (World Cup final rounds)
# Logic: how much does a result in this tournament predict
#        World Cup performance?
# =============================================================

tournament_weights = {
    # ── Tier 5: World Cup (most predictive) ──────────────────
    "FIFA World Cup"                             : 5.0,

    # ── Tier 4: Major continental finals ─────────────────────
    "UEFA Euro"                                  : 4.0,
    "Copa America"                               : 4.0,
    "Africa Cup of Nations"                      : 4.0,
    "AFC Asian Cup"                              : 4.0,
    "CONCACAF Gold Cup"                          : 4.0,
    "FIFA Confederations Cup"                    : 4.0,

    # ── Tier 3: World Cup qualifiers ─────────────────────────
    "FIFA World Cup qualification"               : 3.5,
    "FIFA World Cup qualification (CONMEBOL)"    : 3.5,

    # ── Tier 3: Continental qualifiers ───────────────────────
    "UEFA Euro qualification"                    : 3.0,
    "African Cup of Nations qualification"       : 3.0,
    "AFC Asian Cup qualification"                : 3.0,
    "CONCACAF Championship qualification"        : 3.0,
    "CONCACAF Nations League"                    : 3.0,
    "UEFA Nations League"                        : 3.0,
    "African Nations Championship"               : 3.0,
    "African Nations Championship qualification" : 3.0,
    "OFC Nations Cup"                            : 3.0,
    "OFC Nations Cup qualification"              : 3.0,
    "COSAFA Cup"                                 : 2.5,
    "CECAFA Cup"                                 : 2.5,
    "WAFU Cup of Nations"                        : 2.5,
    "AFF Championship"                           : 2.5,
    "AFF Championship qualification"             : 2.5,
    "SAFF Championship"                          : 2.5,
    "CONCACAF Nations League qualification"      : 2.5,
    "CFU Caribbean Cup"                          : 2.5,
    "CFU Caribbean Cup qualification"            : 2.5,
    "UNCAF Cup"                                  : 2.5,

    # ── Tier 2: Secondary tournaments ────────────────────────
    "Copa Centroamericana"                       : 2.0,
    "Copa Centroamericana qualification"         : 2.0,
    "Pan Arab Games"                             : 2.0,
    "Arab Cup"                                   : 2.0,
    "Arab Cup qualification"                     : 2.0,
    "Gulf Cup"                                   : 2.0,
    "EAFF Championship"                          : 2.0,
    "EAFF Championship qualification"            : 2.0,
    "Intercontinental Play-off"                  : 2.0,
    "AFC Challenge Cup"                          : 2.0,
    "AFC Challenge Cup qualification"            : 2.0,
    "AFC Solidarity Cup"                         : 2.0,
    "King's Cup"                                 : 2.0,
    "Kirin Cup"                                  : 2.0,
    "Merlion Cup"                                : 2.0,
    "Cyprus International Tournament"            : 2.0,
    "Cyprus Cup"                                 : 2.0,
    "Tournoi de France"                          : 2.0,
    "Intercontinental Cup"                       : 2.0,
    "Four Nations Tournament"                    : 2.0,

    # ── Tier 1: Friendlies & everything else → default ───────
    "Friendly"                                   : 1.0,
}

# Any tournament not in the dict gets 1.5
# (minor regional cups — more than a friendly, less than a qualifier)
DEFAULT_WEIGHT = 1.5

matches["tournament_weight"] = (
    matches["tournament"]
    .map(tournament_weights)
    .fillna(DEFAULT_WEIGHT)
)

# Check
print(matches[["tournament", "tournament_weight"]].drop_duplicates()
      .sort_values("tournament_weight", ascending=False).to_string())

print(f"\nNaNs: {matches['tournament_weight'].isna().sum()}")
print(f"Range: {matches['tournament_weight'].min()} → {matches['tournament_weight'].max()}")
print(f"\nWeight distribution:")
print(matches["tournament_weight"].value_counts().sort_index(ascending=False))

                                       tournament  tournament_weight
2373                               FIFA World Cup                5.0
921                                 AFC Asian Cup                4.0
541                                     UEFA Euro                4.0
184                  FIFA World Cup qualification                3.5
21                    AFC Asian Cup qualification                3.0
17652                         UEFA Nations League                3.0
18740                     CONCACAF Nations League                3.0
628          African Cup of Nations qualification                3.0
2562                      UEFA Euro qualification                3.0
6372               AFF Championship qualification                2.5
17640       CONCACAF Nations League qualification                2.5
1488                            CFU Caribbean Cup                2.5
1116              CFU Caribbean Cup qualification                2.5
1009                              

In [9]:
# Check what's sitting at 1.5 that shouldn't be
minor = matches[matches["tournament_weight"] == 1.5]["tournament"].value_counts()
print(minor.to_string())

tournament
African Cup of Nations                        473
Gold Cup                                      328
Island Games                                  268
Copa América                                  248
SAFF Cup                                      135
WAFF Championship                             114
CONIFA World Football Cup                     101
Oceania Nations Cup                            99
Confederations Cup                             96
Pacific Games                                  75
Indian Ocean Island Games                      68
Gold Cup qualification                         67
Viva World Cup                                 60
CONIFA European Football Cup                   49
South Pacific Games                            48
Muratti Vase                                   46
Coupe de l'Outre-Mer                           42
Baltic Cup                                     39
Merdeka Tournament                             33
Nehru Cup                              

In [10]:
# =============================================================
# FIX — patch any major tournaments that fell to default 1.5
# Add to this dict anything important from the list above
# =============================================================

patch = {
    "Copa America"                               : 4.0,
    "Africa Cup of Nations"                      : 4.0,
    "Copa America qualification"                 : 3.0,
    "African Cup of Nations"                     : 4.0,
    "CONCACAF Gold Cup"                          : 4.0,
    "CONCACAF Gold Cup qualification"            : 3.0,
    "OFC Nations Cup"                            : 3.0,
    "FIFA Confederations Cup"                    : 4.0,
    "SAFF Championship"                          : 2.5,
    "Pan Arab Games"                             : 2.0,
    "Arab Cup"                                   : 2.0,
    "OFC Nations Cup qualification"              : 3.0,
}

for tournament, weight in patch.items():
    mask = matches["tournament"] == tournament
    if mask.sum() > 0:
        matches.loc[mask, "tournament_weight"] = weight
        print(f"Patched: {tournament:45s} → {weight}  ({mask.sum()} rows)")
    else:
        print(f"NOT FOUND: {tournament}")

print(f"\nWeight distribution after patch:")
print(matches["tournament_weight"].value_counts().sort_index(ascending=False))

NOT FOUND: Copa America
NOT FOUND: Africa Cup of Nations
NOT FOUND: Copa America qualification
Patched: African Cup of Nations                        → 4.0  (473 rows)
NOT FOUND: CONCACAF Gold Cup
NOT FOUND: CONCACAF Gold Cup qualification
NOT FOUND: OFC Nations Cup
NOT FOUND: FIFA Confederations Cup
NOT FOUND: SAFF Championship
NOT FOUND: Pan Arab Games
Patched: Arab Cup                                      → 2.0  (61 rows)
NOT FOUND: OFC Nations Cup qualification

Weight distribution after patch:
tournament_weight
5.0     384
4.0    1006
3.5    5332
3.0    4467
2.5    1542
2.0     777
1.5    2467
1.0    8020
Name: count, dtype: int64


In [11]:
# =============================================================
# FINAL PATCH — using exact tournament names from the dataset
# =============================================================

patch_exact = {
    # Major continental tournaments → Tier 4
    "Copa América"                  : 4.0,
    "African Cup of Nations"        : 4.0,   # already patched, safe to repeat
    "Gold Cup"                      : 4.0,
    "Confederations Cup"            : 4.0,
    "Oceania Nations Cup"           : 4.0,

    # Qualifiers → Tier 3
    "Gold Cup qualification"        : 3.0,
    "WAFF Championship"             : 3.0,

    # Secondary regional → Tier 2.5
    "SAFF Cup"                      : 2.5,
    "Pacific Games"                 : 2.0,
    "South Pacific Games"           : 2.0,
    "Merdeka Tournament"            : 2.0,
    "Nehru Cup"                     : 2.0,
    "Baltic Cup"                    : 2.0,
    "Amílcar Cabral Cup"            : 2.0,
    "ASEAN Championship"            : 2.0,
    "Indian Ocean Island Games"     : 2.0,
    "FIFA Series"                   : 2.0,

    # Non-FIFA / island games → keep at 1.5 (default already correct)
    # Island Games, CONIFA, Viva World Cup, Muratti Vase etc. stay 1.5
}

for tournament, weight in patch_exact.items():
    mask = matches["tournament"] == tournament
    if mask.sum() > 0:
        matches.loc[mask, "tournament_weight"] = weight
        print(f"✅ Patched: {tournament:40s} → {weight}  ({mask.sum()} rows)")
    else:
        print(f"⚠️  NOT FOUND: {tournament}")

print(f"\nFinal weight distribution:")
print(matches["tournament_weight"].value_counts().sort_index(ascending=False))

print(f"\nStill at 1.5 (minor/non-FIFA — expected):")
print(matches[matches["tournament_weight"] == 1.5]["tournament"]
      .value_counts().head(20).to_string())

✅ Patched: Copa América                             → 4.0  (248 rows)
✅ Patched: African Cup of Nations                   → 4.0  (473 rows)
✅ Patched: Gold Cup                                 → 4.0  (328 rows)
✅ Patched: Confederations Cup                       → 4.0  (96 rows)
✅ Patched: Oceania Nations Cup                      → 4.0  (99 rows)
✅ Patched: Gold Cup qualification                   → 3.0  (67 rows)
✅ Patched: WAFF Championship                        → 3.0  (114 rows)
✅ Patched: SAFF Cup                                 → 2.5  (135 rows)
✅ Patched: Pacific Games                            → 2.0  (75 rows)
✅ Patched: South Pacific Games                      → 2.0  (48 rows)
✅ Patched: Merdeka Tournament                       → 2.0  (33 rows)
✅ Patched: Nehru Cup                                → 2.0  (33 rows)
✅ Patched: Baltic Cup                               → 2.0  (39 rows)
✅ Patched: Amílcar Cabral Cup                       → 2.0  (30 rows)
✅ Patched: ASEAN Championship

In [12]:
# =============================================================
# LABEL — encode outcome as integer
# 0 = away win  |  1 = draw  |  2 = home win
# =============================================================

label_map = {
    "away_win" : 0,
    "draw"     : 1,
    "home_win" : 2,
}

matches["label"] = matches["outcome"].map(label_map)

print("Label distribution:")
print(matches["label"].value_counts().sort_index())
print(f"\nNaNs: {matches['label'].isna().sum()}")

Label distribution:
label
0     6848
1     5603
2    11544
Name: count, dtype: int64

NaNs: 0


In [13]:
# =============================================================
# ASSEMBLE — keep only feature columns + label
# Drop raw columns that the model must never see
# =============================================================

feature_cols = [
    "home_fifa_points",     # Feature 1
    "away_fifa_points",     # Feature 1
    "fifa_points_diff",     # Feature 2
    "home_form",            # Feature 3
    "away_form",            # Feature 3
    "h2h_home_win_rate",    # Feature 4
    "tournament_weight",    # Feature 5
    "home_avg_goal_diff",   # Feature 6
    "away_avg_goal_diff",   # Feature 6
]

# Keep metadata columns so you can debug later — not fed to model
meta_cols = ["date", "home_team", "away_team", "tournament"]

dataset = matches[meta_cols + feature_cols + ["label"]].copy()

print(f"Dataset shape: {dataset.shape}")
print(f"\nSample:")
print(dataset.head(5).to_string())
print(f"\nNaNs per column:")
print(dataset.isna().sum())

Dataset shape: (23995, 14)

Sample:
        date            home_team away_team tournament  home_fifa_points  away_fifa_points  fifa_points_diff  home_form  away_form  h2h_home_win_rate  tournament_weight  home_avg_goal_diff  away_avg_goal_diff  label
0 2000-01-04                Egypt      Togo   Friendly               0.0               0.0               0.0        0.5        0.5                0.5                1.0                 0.0                 0.0      2
1 2000-01-07              Tunisia      Togo   Friendly               0.0               0.0               0.0        0.5        0.0                0.5                1.0                 0.0                -1.0      2
2 2000-01-08  Trinidad And Tobago    Canada   Friendly               0.0               0.0               0.0        0.5        0.5                0.5                1.0                 0.0                 0.0      1
3 2000-01-09         Burkina Faso     Gabon   Friendly               0.0               0.0          

In [14]:
from sklearn.preprocessing import MinMaxScaler

# =============================================================
# NORMALIZE — scale all features to [0, 1]
# MinMaxScaler is safe here: no assumptions about distribution.
# Meta columns and label are NOT scaled.
# =============================================================

scaler = MinMaxScaler()

dataset[feature_cols] = scaler.fit_transform(dataset[feature_cols])

print("Feature ranges after normalization:")
print(dataset[feature_cols].describe().loc[["min", "max"]].to_string())

# =============================================================
# EXPORT
# =============================================================

out_path = "../data/features/feature_dataset.csv"
dataset.to_csv(out_path, index=False)

print(f"\n✅ Phase 2 complete.")
print(f"   Saved to: {out_path}")
print(f"   Shape:    {dataset.shape}")
print(f"\nColumn list fed to Phase 3 model:")
for col in feature_cols:
    print(f"   {col}")
print(f"   label  (0=away win, 1=draw, 2=home win)")

Feature ranges after normalization:
     home_fifa_points  away_fifa_points  fifa_points_diff  home_form  away_form  h2h_home_win_rate  tournament_weight  home_avg_goal_diff  away_avg_goal_diff
min               0.0               0.0               0.0        0.0        0.0                0.0                0.0                 0.0                 0.0
max               1.0               1.0               1.0        1.0        1.0                1.0                1.0                 1.0                 1.0

✅ Phase 2 complete.
   Saved to: ../data/features/feature_dataset.csv
   Shape:    (23995, 14)

Column list fed to Phase 3 model:
   home_fifa_points
   away_fifa_points
   fifa_points_diff
   home_form
   away_form
   h2h_home_win_rate
   tournament_weight
   home_avg_goal_diff
   away_avg_goal_diff
   label  (0=away win, 1=draw, 2=home win)


In [15]:
import pickle

scaler_path = "../data/features/scaler.pkl"

with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)

print(f"✅ Scaler saved to {scaler_path}")
print("   Load it in Phase 3 with:")
print("   with open('data/features/scaler.pkl', 'rb') as f:")
print("       scaler = pickle.load(f)")

✅ Scaler saved to ../data/features/scaler.pkl
   Load it in Phase 3 with:
   with open('data/features/scaler.pkl', 'rb') as f:
       scaler = pickle.load(f)
